In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("clean_customers_sales.csv")

df.shape

(8950, 20)

#Päevade arv viimsest ostust.

Määran analüüsi jaoks viitekuupäeva, mille suhtes arvutatakse kliendi viimase ostu kaugus ajas (Recency).

In [3]:
today = pd.to_datetime("2025-02-28")

recency = df.groupby("customer_id")["sale_date"].max().reset_index()

recency["recency_days"] = (
    today - pd.to_datetime(recency["sale_date"])
).dt.days

recency.head()

,customer_id,sale_date,recency_days
0,2001.0,2024-11-29,91
1,2004.0,2024-12-19,71
2,2005.0,2024-10-03,148
3,2006.0,2023-11-09,477
4,2007.0,2025-01-30,29


#Kogukulutus- iga kliendi kõik ostus liitan total_price kokku.

Frequency näitab, kui sageli klient ettevõttest ostab.

#Mitu ostu iga klient tegi?

Recency näitab, mitu päeva on möödunud kliendi viimasest ostust.
Mida väiksem väärtus, seda aktiivsem klient.

#Monetary arvutamine

Arvutame iga kliendi kogukulutuse.
Monetary näitab kliendi rahalist väärtust ettevõtte jaoks.

In [4]:
frequency = df.groupby("customer_id")["sale_id"].count().reset_index()

frequency.columns = ["customer_id", "frequency"]

frequency.head()

,customer_id,frequency
0,2001.0,2
1,2004.0,2
2,2005.0,4
3,2006.0,1
4,2007.0,1


In [5]:
monetary = df.groupby("customer_id")["total_price"].sum().reset_index()

monetary.columns = ["customer_id", "monetary"]

monetary.head()

,customer_id,monetary
0,2001.0,203.92
1,2004.0,1198.56
2,2005.0,959.60
3,2006.0,327.06
4,2007.0,318.63


#Ühendan Recency, Frequency ja Monetary näitajad ühte tabelisse.
Tulemuseks on üks rida iga kliendi kohta koos kõigi RFM mõõdikutega.

In [6]:
rfm = recency.merge(
    frequency,
    on="customer_id"
).merge(
    monetary,
    on="customer_id"
)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary
0,2001.0,2024-11-29,91,2,203.92
1,2004.0,2024-12-19,71,2,1198.56
2,2005.0,2024-10-03,148,4,959.60
3,2006.0,2023-11-09,477,1,327.06
4,2007.0,2025-01-30,29,1,318.63


In [7]:
rfm.shape

(2540, 5)

#RFM skooride määramine

Jagan kliendid kvintiilidesse ning määran igale mõõdikule skoori vahemikus 1–5.
Kõrgem skoor näitab paremat tulemust.

Recency puhul vastupidine loogika:
hiljutine ost annab kõrgema skoori.

In [8]:
rfm["R_score"] = pd.qcut(
    rfm["recency_days"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    5,
    labels=[1,2,3,4,5]
)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score
0,2001.0,2024-11-29,91,2,203.92,4,1,1
1,2004.0,2024-12-19,71,2,1198.56,4,1,4
2,2005.0,2024-10-03,148,4,959.60,3,4,3
3,2006.0,2023-11-09,477,1,327.06,1,1,1
4,2007.0,2025-01-30,29,1,318.63,5,1,1


#RFM koguskoori arvutamine

Liidan Recency, Frequency ja Monetary skoorid.
Koguskoor võimaldab hinnata kliendi üldist väärtust ühe numbrina.

In [9]:
rfm["RFM_Score"] = (
    rfm["R_score"].astype(int)
    + rfm["F_score"].astype(int)
    + rfm["M_score"].astype(int)
)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score
0,2001.0,2024-11-29,91,2,203.92,4,1,1,6
1,2004.0,2024-12-19,71,2,1198.56,4,1,4,9
2,2005.0,2024-10-03,148,4,959.60,3,4,3,10
3,2006.0,2023-11-09,477,1,327.06,1,1,1,3
4,2007.0,2025-01-30,29,1,318.63,5,1,1,7


#Kliendisegmentide loomine

Jaotan kliendid RFM koguskoori põhjal segmentidesse.
Segmendid aitavad määrata erinevaid turundus- ja kliendihaldusstrateegiaid.
Koostan ülevaate klientide arvust ja osakaalust igas segmendis.
See aitab hinnata kliendibaasi struktuuri ja peamisi sihtgruppe.

In [10]:
def segment_customer(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"

rfm["segment"] = rfm["RFM_Score"].apply(segment_customer)

rfm.head()

,customer_id,sale_date,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,segment
0,2001.0,2024-11-29,91,2,203.92,4,1,1,6,At Risk
1,2004.0,2024-12-19,71,2,1198.56,4,1,4,9,Potential
2,2005.0,2024-10-03,148,4,959.60,3,4,3,10,Loyal
3,2006.0,2023-11-09,477,1,327.06,1,1,1,3,Lost
4,2007.0,2025-01-30,29,1,318.63,5,1,1,7,Potential


In [11]:
segment_summary = (
    rfm["segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = ["segment", "customers"]

segment_summary["percentage"] = (
    segment_summary["customers"] / len(rfm) * 100
).round(2)

segment_summary

,segment,customers,percentage
0,Potential,759,29.88
1,Loyal,679,26.73
2,At Risk,529,20.83
3,VIP Champions,455,17.91
4,Lost,118,4.65


In [12]:
rfm[["R_score", "F_score", "M_score"]].describe()

,R_score,F_score,M_score
count,2540,2540,2540
unique,5,5,5
top,5,1,1
freq,511,508,510


#RFM kokkuvõte

Kliendid segmenteeriti Recency, Frequency ja Monetary väärtuste põhjal.
Skoorid määrati kvintiilide alusel skaalal 1-5. Kliendisegmendid:

Segmendid:
- VIP Champions
- Loyal
- Potential
- At Risk
- Lost


#Jah VIP klientide arv on väga mõistlik. 17,9%. Samuti lost klientide arv on vähe 4,65%, samuti mõistlik. Suurim osa on Potential/Loyal, mis on realistlik.

#Loon uue faili RFM tulemustega, mida kasutan järgmises ülesandes.

In [13]:
rfm.to_csv("rfm_results.csv", index=False)